# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoder-only:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
- Decoder-based:
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)

In [ ]:
!pip install mteb

In [ ]:
import numpy as np

import torch
from datasets import load_dataset
import pandas as pd

import datasets

## Baseline models

In [ ]:
from transformers import AutoTokenizer, AutoModel

In [ ]:
def tokenize_chunking_strategy(tokenizer, inputs, chunk_size, overlap):
    real_chunks_size = chunk_size - tokenizer.num_special_tokens_to_add(pair=False)    # for each chunk special tokens will be appened after

    number_of_chunks = []    # number of chunks for each text in input
    outer_chunked_texts_batch = []    # long batch of all texts chunks

    for text in inputs:
        token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
        start = 0
        chunk_number = 0
        while start < len(token_ids):
            end = start + real_chunks_size
            chunk_tokens = token_ids[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            outer_chunked_texts_batch.append(chunk_text)
            start += real_chunks_size - overlap
            chunk_number += 1
        number_of_chunks.append(chunk_number)

    tokenized_outer_batch = tokenizer(
        outer_chunked_texts_batch,
        add_special_tokens=True,
        max_length=chunk_size,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    return (tokenized_outer_batch, number_of_chunks)

def re_group_chunked_outputs(outputs_last_hidden_state, number_of_chunks):
    # outputs shape: [ num_chunks * num_texts, chunk_size, *]
    # converting to [num_texts, num_chunks, chunk_size, *]
    re_grouped = []
    text_starts_i = 0
    for n_chunks in number_of_chunks:
        text_ends_i = text_starts_i + n_chunks
        re_grouped.append(
            outputs_last_hidden_state[text_starts_i:text_ends_i, :, :]
        )
        text_starts_i = text_ends_i
    return re_grouped

def tokenize_first_startegy(tokenizer, inputs, max_length):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return tokenized


def tokenize_max_tokens_strategy(tokenizer, inputs):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="longest",
        truncation=True,
        max_length=tokenizer.model_max_length,
        return_tensors="pt")
    return tokenized

In [ ]:
from mteb import EncoderProtocol
from mteb.similarity_functions import cos_sim

In [ ]:
from enum import Enum

class Strategy(Enum):
    chunking = "chunking"
    first = "first"
    max_tokens = "max_tokens"

In [ ]:
class XMLRoBERTa(EncoderProtocol):

    name = "xlm-roberta-large"
    similarity = staticmethod(cos_sim)

    def __init__(self, max_size, strategy: Strategy, overlap):
        self.model = AutoModel.from_pretrained(
            "FacebookAI/xlm-roberta-large",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.model.eval()

    def __encode_batch(
            self,
            texts,
            **kwargs) -> np.ndarray:

        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(self.tokenizer, texts, self.max_size, self.overlap)
        if self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        if self.strategy == Strategy.max_tokens:
            tokenized = tokenize_max_tokens_strategy(self.tokenizer, texts)

        with torch.inference_mode():
            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            outputs = self.model(**tokenized)

            if self.strategy == Strategy.chunking:
                re_grouped = re_group_chunked_outputs(outputs.last_hidden_state, numbers_of_chunks)
                embeddings = torch.stack([
                    t.mean(dim=1).mean(dim=0).detach().cpu()
                    for t in re_grouped
                ])
            if (self.strategy == Strategy.first
                or self.strategy == Strategy.max_tokens):
                embeddings = outputs.last_hidden_state.mean(dim=1)


        embeddings = embeddings.detach().cpu().numpy()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]

        texts = [text for batch in inputs for text in batch["text"]]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(batch)
            all_embeddings.append(batch_embeddings)

        return_embeddings = np.vstack(all_embeddings)
        print(return_embeddings.shape)
        return return_embeddings



class Qwen3_Embedding(EncoderProtocol):

    name = "Qwen3-Embedding-0.6B"
    similarity = staticmethod(cos_sim)

    def __init__(self, max_size, strategy, overlap, return_hidden_states=False):
        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            padding_side='left')

        self.model = AutoModel.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.return_hidden_states = return_hidden_states

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states):
        return last_hidden_states[:, -1]

    def preprocess_query(self, texts):
        task = 'Given a search query, retrieve relevant passages that answer the query'
        return f'Instruct: {task}\nQuery:{texts}'

    def __encode_batch(self, texts, **kwargs) -> np.ndarray:
        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(
                self.tokenizer, texts, self.max_size, self.overlap
            )
        elif self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        else:
            tokenized = tokenize_max_tokens_strategy(self.tokenizer, texts)

        with torch.inference_mode():
            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            outputs = self.model(**tokenized)

            if self.strategy == Strategy.chunking:
                re_grouped = re_group_chunked_outputs(
                    outputs.last_hidden_state, numbers_of_chunks
                )
                embeddings = torch.stack([
                    self.__get_eos_token_embedding(t).mean(dim=0)
                    for t in re_grouped
                ])
            else:
                embeddings = self.__get_eos_token_embedding(
                    outputs.last_hidden_state
                )

        embeddings = embeddings.detach().cpu().numpy()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]

        texts = [text for batch in inputs for text in batch["text"]]
        if prompt_type.value == "query":
            texts = [self.preprocess_query(t) for t in texts]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(batch)
            all_embeddings.append(batch_embeddings)

        return_embeddings = np.vstack(all_embeddings)
        print(return_embeddings.shape)
        return return_embeddings

## LongEmbed LEMBWikimQARetrieval

In [ ]:
import mteb

In [ ]:
long_embed_wiki_task = mteb.get_task("LEMBWikimQARetrieval")

In [ ]:
import json
import gc

strategies = [
    {
        "name": "Chunking",
        "strategy": Strategy.chunking,
        "max_size": 512,
        "overlap": 0
    },
    {
        "name": "Chunking 64 Overlap",
        "strategy": Strategy.chunking,
        "max_size": 512,
        "overlap": 64
    },
    {
        "name": "First",
        "strategy": Strategy.first,
        "max_size": 512,
        "overlap": None
    },
]

main_scores_roberta = []
roberta = None

for strategy in strategies:
    roberta = XMLRoBERTa(
        max_size=strategy["max_size"],
        strategy=strategy["strategy"],
        overlap=strategy["overlap"])

    eval = long_embed_wiki_task.evaluate(roberta, encode_kwargs={"batch_size": 4})
    main_scores_roberta.append(eval["default"]["main_score"])
    # save scores, eval is dictionary
    file_name = f"roberta_{strategy['name']}_scores.json"
    with open(file_name, "w") as f:
        json.dump(eval, f)

    # free memory just in case
    del roberta
    torch.cuda.empty_cache()
    gc.collect()

main_scores_qwen3 = []
qwen3 = None

for strategy in strategies:
    qwen3 = Qwen3_Embedding(
        max_size=strategy["max_size"],
        strategy=strategy["strategy"],
        overlap=strategy["overlap"])

    eval = long_embed_wiki_task.evaluate(qwen3, encode_kwargs={"batch_size": 4})
    main_scores_qwen3.append(eval["default"]["main_score"])
    # save scores, eval is dictionary
    file_name = f"qwen3_{strategy['name']}_scores.json"
    with open(file_name, "w") as f:
        json.dump(eval, f)

    # free memory just in case
    del qwen3
    torch.cuda.empty_cache()
    gc.collect()

Filtering queries by qrels:   0%|          | 0/300 [00:00<?, ? examples/s]

Processing queries for dataloading:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 1024)


Converting corpus dict:   0%|          | 0/300 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (4234 > 512). Running this sequence through the model will result in indexing errors


...

Filtering queries by qrels:   0%|          | 0/300 [00:00<?, ? examples/s]

Processing queries for dataloading:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 1024)


Converting corpus dict:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 1024)


In [ ]:
main_scores_roberta

[0.01584, 0.01657, 0.01804]

In [ ]:
main_scores_qwen3

[0.76854, 0.76813, 0.56949]

In [ ]:
pd.DataFrame(
    {
        "Strategy": [s["name"] for s in strategies],
        "Qwen3 score": main_scores_qwen3,
        "XML-RoBERTa score": main_scores_roberta
    }
).set_index("Strategy")

,Qwen3 score,XML-RoBERTa score
Strategy,,
Chunking,0.76854,0.01584
Chunking 64 Overlap,0.76813,0.01657
First,0.56949,0.01804


# Memory module

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
k = torch.tensor([
    [
        [1, 1, 1, 1],
        [2, 2, 2, 2]
    ],
    [
        [1, 1, 1, 1],
        [2, 2, 2, 2]
    ]

],
    dtype=torch.float
)

q = torch.tensor(
    [
        [1, 1, 1, 1],
        [2, 2, 2, 2]
    ],
    dtype=torch.float
)

F.softmax(torch.bmm(k, q.unsqueeze(-1)).squeeze(-1), dim=1).shape

torch.Size([2, 2])

In [ ]:
hidden_state_dim = 4
n_units = 2

W_r = nn.Linear(hidden_state_dim, hidden_state_dim)
U_r = nn.Linear(hidden_state_dim, hidden_state_dim)
W_z = nn.Linear(hidden_state_dim, hidden_state_dim)
U_z = nn.Linear(hidden_state_dim, hidden_state_dim)
W_c = nn.Linear(hidden_state_dim, hidden_state_dim)
U_c = nn.Linear(hidden_state_dim, hidden_state_dim)

memory = torch.tensor([
    [
        [1, 1, 1, 1],
        [2, 2, 2, 2]
    ],
    [
        [1, 1, 1, 1],
        [2, 2, 2, 2]
    ]

],
    dtype=torch.float
)

hidden = torch.tensor(
    [
        [1, 1, 1, 1],
        [2, 2, 2, 2]
    ],
    dtype=torch.float
)

u = U_r(hidden)
w = W_r(memory)

print(u)
print(w)

tensor([[ 0.3101, -0.3275, -0.3576,  0.2642],
        [ 0.6692, -0.4395, -0.9591,  0.3810]], grad_fn=<AddmmBackward0>)
tensor([[[-0.6717, -0.0586, -0.6796,  0.3434],
         [-1.1455, -0.1629, -1.2649,  0.2018]],

        [[-0.6717, -0.0586, -0.6796,  0.3434],
         [-1.1455, -0.1629, -1.2649,  0.2018]]], grad_fn=<ViewBackward0>)


In [ ]:
u + w

tensor([[[-0.3616, -0.3861, -1.0371,  0.6076],
         [-0.4763, -0.6025, -2.2240,  0.5828]],

        [[-0.3616, -0.3861, -1.0371,  0.6076],
         [-0.4763, -0.6025, -2.2240,  0.5828]]], grad_fn=<AddBackward0>)

In [ ]:
class Writer(nn.Module):

    def __init__(self, hidden_state_dim):
        super().__init__()

        self.hidden_state_dim = hidden_state_dim

        # attention Key and Query matrix
        self.W_k = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_q = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)

        # GRU matrices
        self.W_r = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_r = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_z = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_z = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_c = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_c = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)

    def get_attention_weights(self, h, memory):
        """
        h: hidden state (batch_size, hidden_state_dim)
        memory: memory (batch_size, n_units, hidden_state_dim)
        """
        k = self.W_k(memory)    # (batch_size, n_units, hidden_state_dim)
        q = self.W_q(h).unsqueeze(-1)    # (batch_size, hidden_state_dim, 1)
        scores = torch.bmm(k, q).squeeze(-1)    # (batch_size, n_units)
        attention_weights = F.softmax(scores, dim=1)    # (batch_size, n_units)
        return attention_weights
    
    def forward(self, input, memory):
        """
        input: input (batch_size, hidden_state_dim)
        memory: memory (batch_size, n_units, hidden_state_dim)
        """
        # Attention
        attention_weights = self.get_attention_weights(input, memory)
        
        # GRU
        r = F.sigmoid(self.W_r(input) + self.U_r(memory))
        z = F.sigmoid(self.W_z(input) + self.U_z(memory))
        
        m_hat = F.tanh(self.W_c(input) + self.U_c(r * memory))
        m = z * m_hat + (1 - z) * memory
        return m


class Reader(nn.Module):
    def __init__(self, hidden_state_dim):
        super().__init__()

        self.hidden_state_dim = hidden_state_dim

        # attention Key and Query matrix
        self.W_k = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_q = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)

        # GRU matrices
        self.W_r = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_r = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_z = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_z = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_c = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_c = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)

    def get_attention_weights(self, h, memory):
        """
        h: hidden state (batch_size, hidden_state_dim)
        memory: memory (batch_size, n_units, hidden_state_dim)
        """
        k = self.W_k(memory)    # (batch_size, n_units, hidden_state_dim)
        q = self.W_q(h).unsqueeze(-1)    # (batch_size, hidden_state_dim, 1)
        scores = torch.bmm(k, q).squeeze(-1)    # (batch_size, n_units)
        attention_weights = F.softmax(scores, dim=1)    # (batch_size, n_units)
        return attention_weights

    def forward(self, input, memory):
        # Attention
        attention_weights = self.get_attention_weights(input, memory)

        pooled_memory = torch.bmm(memory, attention_weights)

        # GRU
        r = torch.sigmoid(self.W_r(pooled_memory) + self.U_r(input))
        z = torch.sigmoid(self.W_z(pooled_memory) + self.U_z(input))

        h_hat = torch.tanh(self.W_c(pooled_memory) + self.U_c(r * input))
        h = z * h_hat + (1 - z) * input
        return h


class Memory(nn.Module):

    def __init__(self, hidden_state_dim, n_units, batch_size=1):
        super().__init__()
        self.hidden_state_dim = hidden_state_dim
        self.n_units = n_units
        self.batch_size = batch_size
        self.memory_cells = torch.zeros(batch_size, n_units, self.hidden_state_dim)
        
        # reader will be only helpful for the token generation task.
        # for the embedding generation, only the writer will be used essentially.
        self.reader = Reader(hidden_state_dim)
        self.writer = Writer(hidden_state_dim)

    def from_weights(self, weights):
        pass

    def forward(self, input):
        self.memory_cells = self.writer(input, self.memory_cells)
        output = self.reader(input, self.memory_cells)
        return output

In [ ]:
class MemoryPooling:
    
    def __init__(self, memory_size, hidden_size):
        self.memory_size = memory_size
        self.hidden_size = hidden_size
        # Linear projection + normalization
        self.proj = nn.Linear(
            memory_size * hidden_size, hidden_size    # concatenated memory cells
        )
        self.norm = nn.LayerNorm(hidden_size)

class TransformerWithMemory(nn.Module):

    def __init__(
            self,
            tokenizer,
            transformer,
            memory_size,
            head):
        self.tokenizer = tokenizer
        self.transformer = transformer
        # transformr object can return a hidden states for all tokens
        self.head = head
        self.memory_size = memory_size

    def get_hidden_states_per_batch(self, last_hidden_state, batch_size, numbers_of_chunks):
        """
        last_hidden_state:
            (batch_size, numbers_of_chunks, chunk_size, hidden_size)
        """

        padded_hidden_states = []

        for i in range(batch_size):
            hidden_states_batch = last_hidden_state[i, :numbers_of_chunks[i], :]
            padded_hidden_states.append(hidden_states_batch)

        hidden_state_dim = self.transformer.hidden_size
        padding_value = torch.zeros(hidden_state_dim, dtype=torch.float)

        padded_hidden_states = nn.utils.rnn.pad_sequence(
            padded_hidden_states,
            batch_first=True,
            padding_value=padding_value
        )
        return padded_hidden_states
    
    def get_hidden_states(self, input):
        tokenized, numbers_of_chunks = tokenize_chunking_strategy(
            self.tokenizer, input, self.max_size, self.overlap
        )
        outputs = self.transformer(tokenized)

        hidden_states = self.get_hidden_states_per_batch(
            outputs.hidden_states, tokenized.size(0), numbers_of_chunks
        )
        return hidden_states

    def forward(self, input):
        """
        input: (batch_size, seq_len)
        """
        batch_size = input.shape[0]
        self.memory = Memory(
            hidden_state_dim=self.transformer.hidden_size,
            n_units=self.memory_size,
            batch_size=batch_size)

        hidden_states = self.transformer(input)  # (batch_size, seq_len, hidden_size)
        for i in range(len(hidden_states.shape[1])):
            memory(hidden_states[:, i, :])

# Training

The training is done in 2 phases: 
1. The early stage, using the MS MACRO dataset. 
2. The middle stage, using the natural questions dataset.
3. The final stage, using the needle in a haystack dataset.

In [1]:
from datasets import load_dataset

In [2]:
ms_marco_dataset = load_dataset("microsoft/ms_marco", split="train")
print(ms_marco_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ValueError: Config name is missing.
Please pick one among the available configs: ['v1.1', 'v2.1']
Example of usage:
	`load_dataset('microsoft/ms_marco', 'v1.1')`

In [ ]:
class MSMACROCollator:
    """Class for collating MS MARCO scores
    """

    def __init__(self):
        s
